In [3]:
# Sequence ontology file will be used for checking if a feature.type is transcripted or not
# Downloaded from https://github.com/The-Sequence-Ontology/SO-Ontologies
%load_ext autoreload
%autoreload 2

In [4]:
from scripts.canonicalize_genomes import parse_ncbi_assembly
json_path = "../data/hg38/ncbi_dataset/data/GCF_000001405.26/sequence_report.jsonl"
parsed = parse_ncbi_assembly(json_path)
chromosome_names = []
for l in parsed:
    if l["role"] == "assembled-molecule":
        chromosome_names.append(l["refseqAccession"])
        print(l)

{'assemblyAccession': 'GCF_000001405.26', 'assemblyUnit': 'Primary Assembly', 'assignedMoleculeLocationType': 'Chromosome', 'chrName': '1', 'gcCount': '103674491', 'gcPercent': 41.5, 'genbankAccession': 'CM000663.2', 'length': 248956422, 'refseqAccession': 'NC_000001.11', 'role': 'assembled-molecule', 'sequenceName': '1', 'ucscStyleName': 'chr1', 'unlocalizedCount': 9}
{'assemblyAccession': 'GCF_000001405.26', 'assemblyUnit': 'Primary Assembly', 'assignedMoleculeLocationType': 'Chromosome', 'chrName': '2', 'gcCount': '101284083', 'gcPercent': 40.5, 'genbankAccession': 'CM000664.2', 'length': 242193529, 'refseqAccession': 'NC_000002.12', 'role': 'assembled-molecule', 'sequenceName': '2', 'ucscStyleName': 'chr2', 'unlocalizedCount': 2}
{'assemblyAccession': 'GCF_000001405.26', 'assemblyUnit': 'Primary Assembly', 'assignedMoleculeLocationType': 'Chromosome', 'chrName': '3', 'gcCount': '91922884', 'gcPercent': 40.0, 'genbankAccession': 'CM000665.2', 'length': 198295559, 'refseqAccession': 

In [5]:
print(len(chromosome_names))

25


In [6]:
from BCBio import GFF

gff_path = "../data/hg38/ncbi_dataset/data/GCF_000001405.26/genomic.gff"

limit_info = {
    "gff_id": chromosome_names
    #"gff_id": ["NC_000001.11"]
}

records = []

with open(gff_path, "r") as f:
    for rec in GFF.parse(f):
        records.append(rec)

In [21]:
import pronto

sequence_ontology = pronto.Ontology("../so.clean.obo")

# Maps name (like "transcript") to Term object
sequence_name_to_obo_term = {
    term.name: term
    for term in sequence_ontology.terms()
    if term.name is not None and not term.obsolete
}

print(sequence_name_to_obo_term["V_gene_segment"])

Term('SO:0000466', name='V_gene_segment')


In [ ]:
from scripts.canonicalize_genomes import is_feature_type_transcribed
print(is_feature_type_transcribed("pseudogene"))
# skip_deeper_exploration = (
#     "primary_transcript", # Generic category for RNA
#     "mRNA", # Mature RNA 
#     "antisense_RNA", # RNA that's transcribed in the - direction relative to the arbitrary genome orientation
#     "rRNA", # Ribosomal RNA
#     "miRNA", # Micro RNA
#     "snoRNA", # Small nucleolar RNA, guides stuff in mRNA
#     "snRNA", # Small nuclear RNA
#     "tRNA", # Transfer RNA
#     "telomerase_RNA", # The RNA component of the telomerase
#     "RNase_MRP_RNA", # RNA of RNase MRP
#     "RNase_P_RNA", # RNA of ribonuclease P
#     "lnc_RNA", # Long non-coding RNA
# )

False


In [10]:
types = set()

for rec in records:
    for f in rec.features:
        if f.type == "pseudogene":
            for sub_f in f.sub_features:
                types.add(sub_f.type)

print(types)

{'V_gene_segment', 'exon', 'transcript', 'C_gene_segment', 'J_gene_segment'}


In [17]:
ERROR_FEATURES = ("CDS",)
CDJV_SEGMENTS = ("C_gene_segment", "D_gene_segment", "J_gene_segment", "V_gene_segment")

def encountered_error_feature(feature, depth, inside_pseudogene=False):
    if feature.type in CDJV_SEGMENTS:
        return False
    
    if feature.type == "exon" and not inside_pseudogene:
        print(f"Encountered exon (outside pseudogenome) in {depth}")
        return True
    
    if feature.type in ERROR_FEATURES:
        print(f"Encountered {feature.type} on {depth}")
        return True
    
    if feature.type == "pseudogene":
        inside_pseudogenomne = True

    elif not is_feature_type_transcribed(feature.type):
        for sub_f in feature.sub_features:
            if encountered_error_feature(sub_f, depth + 1, inside_pseudogene=inside_pseudogene):
                print(f"Encountered on {feature.type}")
                return True
    
    return False

for rec in records:
    should_break = False
    for f in rec.features:
        if encountered_error_feature(f, 0):
            should_break = True
            break
    if should_break:
        break